# Cohort Definition — Salaried Gen Z, Considered Mutual Funds, Currently Non-Holding

Research question: *"What investment barriers are reported by salaried Gen Z respondents who have considered mutual funds but do not currently hold them?"*

This is a respondent-level SEBI Investor Survey 2025 analysis — not an INDmoney conversion analysis, and not a confirmed first-SIP sample. See `docs/cohort_definition.md` for the full write-up and `docs/barrier_coverage.md` for the barrier-field audit; this notebook produces and verifies both.

**Stops after validating the sample.** No dashboard charts, no barrier rankings, no conclusions about leading barriers here.

## 1. Load data (reusing the verified approach from `01_data_inspection.ipynb`)

Same `python_calamine` load, same two-header-row convention (row 1 = codes, row 2 = descriptions), same original files — nothing in `data/raw/` is modified.

In [1]:
from pathlib import Path
import pandas as pd
from python_calamine import CalamineWorkbook

RAW_DIR = Path("../data/raw")
RESPONDENT_FILE = RAW_DIR / "Respondent Data.XLSX"

wb = CalamineWorkbook.from_path(str(RESPONDENT_FILE))
sheet = wb.get_sheet_by_name(wb.sheet_names[0])
data = sheet.to_python(skip_empty_area=True)

codes, descriptions = data[0], data[1]
rows = data[2:]
df = pd.DataFrame(rows, columns=codes)
desc_of = dict(zip(codes, descriptions))

n_total = len(df)
print("Loaded respondent workbook:", df.shape, "records x columns")
assert df.shape == (109430, 448), "Unexpected shape vs. 01_data_inspection.ipynb — investigate before continuing"

Loaded respondent workbook: (109430, 448) records x columns


## 2. Occupation classification (`Q14`) — per the official SEBI Main Report

Source of truth: the **SEBI Investor Survey 2025 Main Report** PDF (official link in `docs/data_sources.md`), Annexure pp. 104-106, which defines the exact occupation buckets SEBI itself uses (e.g. Figure 4.5). Fetched and cross-checked against the raw PDF text — not assumed from the label text alone. Full reasoning for every category is in `docs/cohort_definition.md` §2; two results are surprising enough to highlight here: SEBI's own Annexure puts **Teacher** and standalone **Doctor** under **Self Employed**, not Salaried, and puts **Skilled/Unskilled Worker** in their own buckets, separate from Salaried.

In [2]:
# Documented "Salaried" set, verbatim from the Annexure (p.105)
SALARIED_DOCUMENTED = {
    "Clerk / Salesman",
    "Supervisory Level",
    "Officer / Executive - Junior",
    "Officer / Executive - Middle /Senior",
    "Postman",
    "Service (Rural In any village) & CWE Education illiterate to 9th standard",
    "Service (Rural In any village) & CWE Education 10 to Graduate",
    "Service (Rural In any village) & CWE Education Grad/Post Grad Prof or Post Grad General",
    "Service (Urban) & CWE Education Grad/Post Grad Prof or Post Grad General",
}
BUSINESS_DOCUMENTED = {
    "Petty trader- street vendor, drivers owning vehicles etc.",
    "Shop Owner- operate from a permanent establishment e.g. wholesalers, distributors etc.",
    "Businessmen/Industrialist with no employees under him/her",
    "Businessmen/Industrialist with 9 or less employees under him/her",
    "Trader / Shopkeeper",
}
SELF_EMPLOYED_DOCUMENTED = {
    "Self-employed professional like Doctors, Lawyers etc.",
    "Teacher",
    "Doctor",
    "Self Employed Professional",
}
AGRICULTURE_DOCUMENTED = {
    "Owner Farmer", "Leased Farmer", "Agricultural Worker",
    "Owner Of Livestock", "Owner Of Fisheries", "Owner Of Poultry",
}
UNSKILLED_DOCUMENTED = {
    "Unskilled worker like Cleaner/housemaids etc.",
    "Unskilled Labourer (Other Than Agriculture)",
}
SKILLED_DOCUMENTED = {
    "Skilled worker like electrician/Mechanic etc.",
    "Artisan / Skilled Labourer",
}
NON_WORKER_DIRECT = {"Student", "Homemaker / Housewife", "Unemployed", "Retired"}

# Not literally named in the Annexure anywhere, despite an obvious sibling category that IS
# named. Flagged explicitly rather than silently folded in.
BUSINESS_INFERRED_GAP = {"Businessmen/Industrialist with 10 or more employees under him/her"}
DOCUMENTATION_GAP_AMBIGUOUS = {
    "Service (Urban) & CWE Education 10 to Graduate",
    "Service (Urban) & CWE Education illiterate to 9th standard",
}
CATCHALL_AMBIGUOUS = {"Others (Specify)"}


def classify_occupation(q14: str) -> str:
    if q14 in SALARIED_DOCUMENTED:
        return "included_salaried_documented"
    if q14 in BUSINESS_DOCUMENTED or q14 in BUSINESS_INFERRED_GAP:
        return "excluded_business"
    if q14 in SELF_EMPLOYED_DOCUMENTED:
        return "excluded_self_employed"
    if q14 in AGRICULTURE_DOCUMENTED:
        return "excluded_agriculture"
    if q14 in UNSKILLED_DOCUMENTED:
        return "excluded_unskilled_worker"
    if q14 in SKILLED_DOCUMENTED:
        return "excluded_skilled_worker"
    if q14 in NON_WORKER_DIRECT:
        return "excluded_non_worker"
    if q14 in DOCUMENTATION_GAP_AMBIGUOUS:
        return "ambiguous_documentation_gap"
    if q14 in CATCHALL_AMBIGUOUS:
        return "ambiguous_catchall"
    return "ambiguous_unclassified"  # safety net — should never fire; checked below

In [3]:
mains_complete = df["MAIN_COMP_STATUS"] == "Main Complete"
gen_z = df["Life_Stage"] == "Gen Z"
mains_genz = df[mains_complete & gen_z]
print("Mains-complete Gen Z respondents:", len(mains_genz))

occ_class = mains_genz["Q14"].apply(classify_occupation)
n_unclassified = (occ_class == "ambiguous_unclassified").sum()
print("Unclassified Q14 values (should be 0):", n_unclassified)
if n_unclassified:
    print("UNCLASSIFIED VALUES:", mains_genz.loc[occ_class == "ambiguous_unclassified", "Q14"].unique())
assert n_unclassified == 0, "Every observed Q14 category must be explicitly classified"

occ_table = (
    pd.DataFrame({"Q14": mains_genz["Q14"], "class": occ_class})
    .groupby(["class", "Q14"]).size().reset_index(name="n")
    .sort_values(["class", "n"], ascending=[True, False])
)
print(f"\n{len(occ_table)} distinct Q14 categories observed within Mains-complete Gen Z respondents:\n")
print(occ_table.to_string(index=False))
print("\nTotals by class:")
print(occ_class.value_counts())
assert occ_class.value_counts().sum() == len(mains_genz)

Mains-complete Gen Z respondents: 24576
Unclassified Q14 values (should be 0): 0

36 distinct Q14 categories observed within Mains-complete Gen Z respondents:

                       class                                                                                     Q14    n
          ambiguous_catchall                                                                        Others (Specify) 1320
 ambiguous_documentation_gap                                          Service (Urban) & CWE Education 10 to Graduate   13
 ambiguous_documentation_gap                              Service (Urban) & CWE Education illiterate to 9th standard    5
        excluded_agriculture                                                                            Owner Farmer  973
        excluded_agriculture                                                                     Agricultural Worker  541
        excluded_agriculture                                                                           Lease

## 3. Multi-select tokenizer for product fields (`Q21A`, `Q22A_All`, `Q23A`, `Q24A`, `Q25A`)

Per `docs/data_inspection.md`, a bare `.str.split(",")` is unsafe — one answer option's own label contains commas (`"Cryptocurrency (e.g. Tether, Bitcoin, Ethereum, etc)"`). The one known comma-bearing label is protected before splitting, then restored, so it round-trips as a single token rather than three fake ones.

Mutual Funds and ETF are always kept as separate tokens; the combined `MF_ETF`/`MF+ETF` tag is preserved as its own distinct token too — **never used as a stand-in for either product** (checked explicitly below).

In [4]:
CRYPTO_ORIG = "Cryptocurrency (e.g. Tether, Bitcoin, Ethereum, etc)"
CRYPTO_PLACEHOLDER = "Cryptocurrency (e.g. Tether\x00Bitcoin\x00Ethereum\x00etc)"
MF_TOKEN = "Mutual Funds (One-time Lumpsum / SIP)"
ETF_TOKEN = "Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF)"

KNOWN_VOCAB = {
    "Alternate Investment Fund (AIF)", "Chit Fund", "Corporate Bonds", CRYPTO_ORIG,
    "Employees Provident Fund (EPF)", ETF_TOKEN,
    "Fixed Deposits / Recurring Deposit / Bank Savings Account", "Futures & Options (F&O)",
    "Gold - Physical form / Sovereign Gold Bond (SGB)", "Life insurance / Unit Linked Insurance Plans (ULIPS)",
    "MF+ETF", "MF_ETF", MF_TOKEN, "National Pension System (NPS)",
    "None of the above", "Not Answered",
    "Post office savings / Kisan Vikas Patra (KVP) / National Savings Certificate (NSC)",
    "Public Provident Fund (PPF) / Voluntary Provident Fund (VPF)",
    "Real Estate Investment Trusts (REITs) and /or Infrastructure Investment Trusts (InvIT)",
    "Real Estate as an Investment (excluding where you are staying)", "Stocks / Shares",
}


def tokenize(raw: str):
    """Split a multi-select cell into known-vocabulary tokens.
    Returns (tokens: list[str], fully_resolved: bool). An empty string yields ([], True) —
    callers must handle blank vs. non-blank status themselves; a blank is never a claim
    about product holding/consideration."""
    if raw == "" or raw is None:
        return [], True
    protected = raw.replace(CRYPTO_ORIG, CRYPTO_PLACEHOLDER)
    tokens = [p.strip().replace("\x00", ", ") for p in protected.split(",")]
    return tokens, all(t in KNOWN_VOCAB for t in tokens)


def tokset(raw: str) -> frozenset:
    tokens, ok = tokenize(raw)
    assert ok, f"Unresolved fragment(s) in: {raw!r}"
    return frozenset(tokens)


# Validate: every distinct non-blank value across all 5 product fields fully resolves
PRODUCT_COLS = ["Q21A", "Q22A_All", "Q23A", "Q24A", "Q25A"]
for col in PRODUCT_COLS:
    s = df[col]
    nonblank_uniques = s[s != ""].unique()
    unresolved = [v for v in nonblank_uniques if not tokenize(v)[1]]
    print(f"{col}: {len(nonblank_uniques):6,d} distinct non-blank values, {len(unresolved)} unresolved")
    assert not unresolved, f"{col} has unresolved fragments: {unresolved[:5]}"

Q21A: 22,393 distinct non-blank values, 0 unresolved
Q22A_All:  2,840 distinct non-blank values, 0 unresolved
Q23A:    103 distinct non-blank values, 0 unresolved


Q24A:     90 distinct non-blank values, 0 unresolved
Q25A:    129 distinct non-blank values, 0 unresolved


In [5]:
# Small, explicit parsing checks — MF only, ETF only, both, combined tag, missing, and the
# internal-comma option — asserted against hand-built cases, not just real data rows.
test_cases = {
    "MF only": (f"{MF_TOKEN},MF_ETF", {MF_TOKEN, "MF_ETF"}),
    "ETF only": (f"{ETF_TOKEN},MF_ETF", {ETF_TOKEN, "MF_ETF"}),
    "both MF and ETF": (f"{MF_TOKEN},{ETF_TOKEN},MF_ETF", {MF_TOKEN, ETF_TOKEN, "MF_ETF"}),
    "combined tag alone would be ambiguous if it ever occurred": ("MF_ETF", {"MF_ETF"}),
    "explicit none": ("None of the above", {"None of the above"}),
    "explicit not answered": ("Not Answered", {"Not Answered"}),
    "blank": ("", set()),
    "option containing internal commas, alone": (CRYPTO_ORIG, {CRYPTO_ORIG}),
    "option containing internal commas, combined with another": (
        f"Stocks / Shares,{CRYPTO_ORIG}", {"Stocks / Shares", CRYPTO_ORIG}
    ),
}
for label, (raw, expected) in test_cases.items():
    tokens, ok = tokenize(raw)
    got = set(tokens)
    status = "OK" if (ok and got == expected) else "FAIL"
    print(f"[{status}] {label}: tokenize({raw!r}) -> {sorted(got)}")
    assert ok and got == expected, f"parsing check failed for: {label}"

# Confirm the combined tag is never a stand-in for MF/ETF in the real data: tag present
# should exactly equal (MF token present OR ETF token present), for every product column.
for col in PRODUCT_COLS:
    tag_name = "MF+ETF" if col == "Q22A_All" else "MF_ETF"
    toks = df[col].apply(tokset)
    has_tag = toks.apply(lambda t: tag_name in t)
    has_mf_or_etf = toks.apply(lambda t: (MF_TOKEN in t) or (ETF_TOKEN in t))
    mismatches = (has_tag != has_mf_or_etf).sum()
    etf_only_but_tagged = (has_tag & toks.apply(lambda t: ETF_TOKEN in t and MF_TOKEN not in t)).sum()
    print(f"{col}: tag<->(mf_or_etf) mismatches = {mismatches} (must be 0); "
          f"tagged rows that are ETF-only (no MF) = {etf_only_but_tagged}")
    assert mismatches == 0

[OK] MF only: tokenize('Mutual Funds (One-time Lumpsum / SIP),MF_ETF') -> ['MF_ETF', 'Mutual Funds (One-time Lumpsum / SIP)']
[OK] ETF only: tokenize('Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF),MF_ETF') -> ['Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF)', 'MF_ETF']
[OK] both MF and ETF: tokenize('Mutual Funds (One-time Lumpsum / SIP),Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF),MF_ETF') -> ['Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF)', 'MF_ETF', 'Mutual Funds (One-time Lumpsum / SIP)']
[OK] combined tag alone would be ambiguous if it ever occurred: tokenize('MF_ETF') -> ['MF_ETF']
[OK] explicit none: tokenize('None of the above') -> ['None of the above']
[OK] explicit not answered: tokenize('Not Answered') -> ['Not Answered']
[OK] blank: tokenize('') -> []
[OK] option containing internal commas, alone: tokenize('Cryptocurrency (e.g. Tether, Bitcoin, Ethereum, etc)') -> ['Cryptocurrency (e.g. Tether, 

Q21A: tag<->(mf_or_etf) mismatches = 0 (must be 0); tagged rows that are ETF-only (no MF) = 1017


Q22A_All: tag<->(mf_or_etf) mismatches = 0 (must be 0); tagged rows that are ETF-only (no MF) = 296


Q23A: tag<->(mf_or_etf) mismatches = 0 (must be 0); tagged rows that are ETF-only (no MF) = 757


Q24A: tag<->(mf_or_etf) mismatches = 0 (must be 0); tagged rows that are ETF-only (no MF) = 545


Q25A: tag<->(mf_or_etf) mismatches = 0 (must be 0); tagged rows that are ETF-only (no MF) = 1225


## 4. Selection funnel

Five steps, each applied to the previous step's output. `QFL` (Investor/Non-Investor) is never used as a stand-in for Mains participation or for mutual-fund ownership — steps 1 and 5 use `MAIN_COMP_STATUS` and `Q22A_All` directly.

In [6]:
funnel = []


def record_step(name, entering_n, retained_n, excluded_n, unknown_n):
    assert retained_n + excluded_n + unknown_n == entering_n, (name, entering_n, retained_n, excluded_n, unknown_n)
    funnel.append({
        "step": name, "entering": entering_n, "retained": retained_n,
        "excluded": excluded_n, "unknown_or_ambiguous": unknown_n,
    })


# Step 1: completed Mains — clear yes/no, no ambiguity
step1_mask = mains_complete
record_step("1. Completed Mains (MAIN_COMP_STATUS == 'Main Complete')",
            n_total, int(step1_mask.sum()), int((~step1_mask).sum()), 0)
step1 = df[step1_mask]

# Step 2: Gen Z — clear yes/no, no ambiguity
step2_mask = step1["Life_Stage"] == "Gen Z"
record_step("2. Gen Z (Life_Stage == 'Gen Z')",
            len(step1), int(step2_mask.sum()), int((~step2_mask).sum()), 0)
step2 = step1[step2_mask]

# Step 3: salaried — documented-included vs. documented-excluded vs. ambiguous (kept out)
occ_class_step2 = step2["Q14"].apply(classify_occupation)
step3_mask = occ_class_step2 == "included_salaried_documented"
step3_ambiguous_mask = occ_class_step2.isin(["ambiguous_documentation_gap", "ambiguous_catchall"])
record_step("3. Salaried, documented (Q14 in SALARIED_DOCUMENTED)",
            len(step2), int(step3_mask.sum()),
            int((~step3_mask & ~step3_ambiguous_mask).sum()), int(step3_ambiguous_mask.sum()))
step3 = step2[step3_mask]  # == the "broader confirmed salaried Gen Z group"

# Step 4: considers mutual funds via Q23A — blank Q23A is "unknown" (not administered),
# not "does not consider"
q23_raw = step3["Q23A"]
q23_tok = q23_raw.apply(tokset)
q23_blank = q23_raw == ""
step4_mask = q23_tok.apply(lambda t: MF_TOKEN in t)
step4_excluded_mask = (~step4_mask) & (~q23_blank)
record_step("4. Considers mutual funds (Q23A contains MF token)",
            len(step3), int(step4_mask.sum()), int(step4_excluded_mask.sum()), int(q23_blank.sum()))
step4 = step3[step4_mask]

# Step 5: interpretable Q22A_All that does not include mutual funds — blank/"Not Answered"
# is "unknown", never treated as "does not hold"
q22_raw = step4["Q22A_All"]
q22_interpretable = (q22_raw != "") & (q22_raw != "Not Answered")
q22_tok = q22_raw.apply(tokset)
holds_mf = q22_tok.apply(lambda t: MF_TOKEN in t)
step5_mask = q22_interpretable & ~holds_mf
step5_excluded_mask = q22_interpretable & holds_mf
step5_unknown_mask = ~q22_interpretable
record_step("5. Interpretable Q22A_All, no MF token (does not currently hold MF)",
            len(step4), int(step5_mask.sum()), int(step5_excluded_mask.sum()), int(step5_unknown_mask.sum()))
step5 = step4[step5_mask]  # == the "focused group"

broader_group = step3
focused_group = step5

funnel_df = pd.DataFrame(funnel)
print(funnel_df.to_string(index=False))
print(f"\nBroader confirmed salaried Gen Z group (within Mains): n = {len(broader_group):,}")
print(f"Focused group (considers MF, does not currently hold it): n = {len(focused_group):,}")

                                                               step  entering  retained  excluded  unknown_or_ambiguous
           1. Completed Mains (MAIN_COMP_STATUS == 'Main Complete')    109430     53357     56073                     0
                                   2. Gen Z (Life_Stage == 'Gen Z')     53357     24576     28781                     0
               3. Salaried, documented (Q14 in SALARIED_DOCUMENTED)     24576      4346     18892                  1338
                 4. Considers mutual funds (Q23A contains MF token)      4346       553      2901                   892
5. Interpretable Q22A_All, no MF token (does not currently hold MF)       553       553         0                     0

Broader confirmed salaried Gen Z group (within Mains): n = 4,346
Focused group (considers MF, does not currently hold it): n = 553


## 5. Integrity checks

Focused group must be a strict subset of the broader group; both groups' identifiers must stay unique; and every focused-group record must independently satisfy all five documented filters (re-derived from the raw columns, not just trusted from the funnel construction above).

In [7]:
# Focused group is a subset of the broader group
assert set(focused_group["Resp_ID_DP"]).issubset(set(broader_group["Resp_ID_DP"]))
print("Focused-group Resp_ID_DP values are all present in the broader group: OK")

# Identifier uniqueness within each group (and workbook-wide, already known from 01, re-checked here)
for name, group in [("broader_group", broader_group), ("focused_group", focused_group), ("full workbook", df)]:
    for id_col in ["Resp_ID_DP", "UniqueId_DP"]:
        n_dup = group[id_col].duplicated().sum()
        n_missing = group[id_col].isna().sum()
        print(f"{name:15s} {id_col:12s} missing={n_missing} duplicated={n_dup} distinct={group[id_col].nunique()} n={len(group)}")
        assert n_dup == 0 and n_missing == 0

# Every focused-group record independently satisfies all 5 filters, re-derived fresh
recheck = (
    (focused_group["MAIN_COMP_STATUS"] == "Main Complete")
    & (focused_group["Life_Stage"] == "Gen Z")
    & (focused_group["Q14"].isin(SALARIED_DOCUMENTED))
    & (focused_group["Q23A"].apply(tokset).apply(lambda t: MF_TOKEN in t))
    & (focused_group["Q22A_All"] != "")
    & (focused_group["Q22A_All"] != "Not Answered")
    & (~focused_group["Q22A_All"].apply(tokset).apply(lambda t: MF_TOKEN in t))
)
print(f"\nRecheck: {recheck.sum()} / {len(focused_group)} focused-group records satisfy all 5 filters independently")
assert recheck.all(), "Every focused-group record must satisfy all 5 filters"

Focused-group Resp_ID_DP values are all present in the broader group: OK
broader_group   Resp_ID_DP   missing=0 duplicated=0 distinct=4346 n=4346
broader_group   UniqueId_DP  missing=0 duplicated=0 distinct=4346 n=4346
focused_group   Resp_ID_DP   missing=0 duplicated=0 distinct=553 n=553
focused_group   UniqueId_DP  missing=0 duplicated=0 distinct=553 n=553
full workbook   Resp_ID_DP   missing=0 duplicated=0 distinct=109430 n=109430
full workbook   UniqueId_DP  missing=0 duplicated=0 distinct=109430 n=109430

Recheck: 553 / 553 focused-group records satisfy all 5 filters independently


## 6. Previous mutual-fund investment within the focused group

The focused group is defined by **current** non-holding only. Past MF investors are kept in, not removed — but tracked separately via `Q24A` ("ever invested in these products in the past"), so the group is never described as uniformly "never invested."

In [8]:
q24_raw = focused_group["Q24A"]
q24_tok = q24_raw.apply(tokset)
q24_blank = q24_raw == ""
prev_mf_investor = q24_tok.apply(lambda t: MF_TOKEN in t)
explicit_no_prev = q24_raw == "None of the above"
other_prev_nonmf = (~q24_blank) & (~prev_mf_investor) & (~explicit_no_prev)


def prev_investment_class(is_blank, is_mf, is_explicit_none, is_other):
    if is_blank:
        return "unknown_blank"
    if is_mf:
        return "past_mf_investor"
    if is_explicit_none:
        return "explicit_no_prior_investment"
    if is_other:
        return "past_investor_other_product_only"
    raise AssertionError


prev_class = pd.Series(
    [prev_investment_class(b, m, n, o) for b, m, n, o in zip(q24_blank, prev_mf_investor, explicit_no_prev, other_prev_nonmf)],
    index=focused_group.index, name="prev_mf_investment_class",
)
print(prev_class.value_counts())
assert prev_class.value_counts().sum() == len(focused_group)
print(
    "\nNote: this group mixes past MF investors (candidate lapsers), respondents with no prior "
    "investment at all, and respondents who invested elsewhere but never in MF — it is NOT a "
    "'first-time SIP non-completers' sample. See docs/cohort_definition.md §5 and "
    "docs/barrier_coverage.md for the official 'Lapser' definition."
)

prev_mf_investment_class
explicit_no_prior_investment        382
past_mf_investor                    136
past_investor_other_product_only     35
Name: count, dtype: int64

Note: this group mixes past MF investors (candidate lapsers), respondents with no prior investment at all, and respondents who invested elsewhere but never in MF — it is NOT a 'first-time SIP non-completers' sample. See docs/cohort_definition.md §5 and docs/barrier_coverage.md for the official 'Lapser' definition.


## 7. Barrier-question coverage audit within the focused group

Coverage only — **no ranking, no treating a blank as "no barrier"**. Full source citations (the SEBI Main Report's own Investor/Non-Investor chapter base counts) are in `docs/barrier_coverage.md`; this cell reproduces the counts that document was built from.

In [9]:
BARRIER_COLS = ["A11_D11", "A12_D12", "A13_D13", "A14_D14", "A15_D15", "AA1_DD1", "AA2_DD2", "AA3_DD3", "AA4_DD4"]

print("Workbook-wide non-blank counts (used to match against SEBI report table bases):")
for c in BARRIER_COLS:
    print(f"  {c:10s} {int((df[c] != '').sum()):>6,d}   {desc_of[c]}")

print("\nCoverage within the focused group (n =", len(focused_group), "):")
coverage_rows = []
for c in BARRIER_COLS:
    s = focused_group[c]
    non_missing = int((s != "").sum())
    missing = int((s == "").sum())
    coverage_rows.append({"column": c, "non_missing": non_missing, "missing": missing})
coverage_df = pd.DataFrame(coverage_rows)
print(coverage_df.to_string(index=False))
assert (coverage_df["non_missing"] + coverage_df["missing"] == len(focused_group)).all()

# Confirm (not merely assume) why A11-A14 are almost all blank here: the few respondents who
# do have an answer hold ETF (routed via the combined MF+ETF holding flag), even though none
# hold mutual funds specifically — consistent with the workbook's {_1_2} combined-slot routing.
a13_nonblank_idx = focused_group.index[focused_group["A13_D13"] != ""]
holds_etf_among_them = focused_group.loc[a13_nonblank_idx, "Q22A_All"].apply(tokset).apply(lambda t: ETF_TOKEN in t)
print(f"\nOf {len(a13_nonblank_idx)} focused-group respondents with a non-blank A13_D13, "
      f"{holds_etf_among_them.sum()} hold ETF (routing confirmed, not just assumed).")

Workbook-wide non-blank counts (used to match against SEBI report table bases):
  A11_D11    13,862   A11_D11:MF+ETF - Frequently you invest in MF/ETF.
  A12_D12    13,862   A12_D12: MF+ETF - What you think are the expected returns for Mutual Funds
  A13_D13    13,862   A13_D13:MF+ETF - What challenges do you face before/ while making fresh investment in MF/ETF
  A14_D14    13,862   A14_D14:  What challenges do you face after making investment in Mutual Funds
  A15_D15     5,710   A15_D15: You have not invested in MF/ETF in the last 1 year. Top 3 Reasons are for not investing
  AA1_DD1     3,168   AA1_DD1:MF+ETF - Top 3 Primary reasons for considering investing in MF/ETFs.
  AA2_DD2    18,223   AA2_DD2:MF+ETF - Top 3 Reasons for not investing in MF/ETF
  AA3_DD3    18,223   AA3_DD3:MF+EF - Factors would encourage you to consider investing in MF/ETF that you currently do not invest in
  AA4_DD4     1,381   AA4_DD4: What were the TOp 3 reasons you stopped investing in MF/ETF.

Coverage w

## 8. Save respondent-level working extracts (`data/processed/`, gitignored)

Both extracts preserve original identifiers, both weight columns, the relevant original (raw, unmodified) answer strings, and the derived selection flags used above. No missing answer is filled with an assumed value — blanks are saved as empty strings, exactly as read. Nothing is written to `public/data/`.

**Correction (post-hoc):** an earlier version of this cell saved a boolean `holds_mf_token` column computed as "does the tokenized `Q22A_All` contain the MF token," with no gate on interpretability. For a blank or `"Not Answered"` `Q22A_All`, the tokenized set is also empty/non-matching, so that boolean silently evaluated to `False` — indistinguishable from a genuine "does not hold MF." That column is removed. A safe, explicit three-state `mf_holding_status` (`holds` / `does_not_hold` / `unknown`) replaces it below, gated on `Q22A_All_status` first. This does **not** change cohort membership: the funnel's own step-5 filter (above) already required `Q22A_All` to be interpretable *before* checking for the MF token, so the 4,346/553 counts are unaffected — verified explicitly after this fix in `analysis/04_segment_comparisons.ipynb`.

In [10]:
def q22_status(raw):
    if raw == "":
        return "blank_not_administered"
    if raw == "Not Answered":
        return "explicit_not_answered"
    return "interpretable"


def mf_holding_status(raw):
    """Three-state, safe-to-reuse MF holding status. Token absence alone never establishes
    non-holding — interpretability is checked first. The combined MF+ETF tag is not used as
    a stand-in for MF (validated in Sec. 3: tag present iff MF token or ETF token present,
    with 0 unresolved fragments across the full vocabulary), so an interpretable answer
    that resolves fully but omits the MF token is a genuine, non-ambiguous "does_not_hold"."""
    status = q22_status(raw)
    if status != "interpretable":
        return "unknown"
    return "holds" if MF_TOKEN in tokset(raw) else "does_not_hold"


def build_common_columns(group):
    out = pd.DataFrame(index=group.index)
    out["Resp_ID_DP"] = group["Resp_ID_DP"]
    out["UniqueId_DP"] = group["UniqueId_DP"]
    out["WeightMainM2"] = group["WeightMainM2"]
    out["Weight_to_Sample"] = group["Weight_to_Sample"]
    out["INT_TYPE"] = group["INT_TYPE"]
    out["MAIN_COMP_STATUS"] = group["MAIN_COMP_STATUS"]
    out["QLISTMAIN"] = group["QLISTMAIN"]
    out["QFL_reference_only_not_a_filter"] = group["QFL"]
    out["Life_Stage"] = group["Life_Stage"]
    out["Q14_occupation_raw"] = group["Q14"]
    out["occupation_class"] = group["Q14"].apply(classify_occupation)
    out["Q21A_awareness_raw"] = group["Q21A"]
    out["Q22A_All_holdings_raw"] = group["Q22A_All"]
    out["Q22A_All_status"] = group["Q22A_All"].apply(q22_status)
    out["mf_holding_status"] = group["Q22A_All"].apply(mf_holding_status)
    out["Q23A_consideration_raw"] = group["Q23A"]
    out["considers_mf_token"] = group["Q23A"].apply(tokset).apply(lambda t: MF_TOKEN in t)
    out["Q24A_past_investment_raw"] = group["Q24A"]
    out["Q25A_never_consider_raw"] = group["Q25A"]
    out["in_focused_group"] = group.index.isin(focused_group.index)
    return out


PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

broader_extract = build_common_columns(broader_group)
broader_path = PROCESSED_DIR / "cohort_salaried_genz_mains.csv"
broader_extract.to_csv(broader_path, index=False)
print("wrote", broader_path.resolve(), "-", broader_extract.shape)

focused_extract = build_common_columns(focused_group)
focused_extract["prev_mf_investment_class"] = prev_class
for c in BARRIER_COLS:
    focused_extract[f"{c}_raw"] = focused_group[c]
focused_path = PROCESSED_DIR / "cohort_focused_considered_mf_not_holding.csv"
focused_extract.to_csv(focused_path, index=False)
print("wrote", focused_path.resolve(), "-", focused_extract.shape)

wrote /Users/keshavdubey/Downloads/Work/Monsoon 2026/Market Research & Validation/Mutual Funds Analysis/data/processed/cohort_salaried_genz_mains.csv - (4346, 20)
wrote /Users/keshavdubey/Downloads/Work/Monsoon 2026/Market Research & Validation/Mutual Funds Analysis/data/processed/cohort_focused_considered_mf_not_holding.csv - (553, 30)


## 9. Final verification summary

Re-states the checks required before this sample can be used further: focused ⊆ broader, unique IDs, every included record satisfies its filters, missing/ambiguous handled explicitly (never silently converted to a substantive answer), and outputs above are aggregate-only (no respondent row was printed anywhere in this notebook).

In [11]:
print("=" * 70)
print("SELECTION FUNNEL")
print("=" * 70)
print(funnel_df.to_string(index=False))

print("\n" + "=" * 70)
print("OCCUPATION CLASSIFICATION TOTALS (within Mains-complete Gen Z)")
print("=" * 70)
print(occ_class.value_counts())

print("\n" + "=" * 70)
print("PREVIOUS MF INVESTMENT WITHIN FOCUSED GROUP")
print("=" * 70)
print(prev_class.value_counts())

print("\n" + "=" * 70)
print("BARRIER COVERAGE WITHIN FOCUSED GROUP")
print("=" * 70)
print(coverage_df.to_string(index=False))

print("\n" + "=" * 70)
print("INTEGRITY CHECKS")
print("=" * 70)
checks = {
    "focused_group ⊆ broader_group": set(focused_group["Resp_ID_DP"]).issubset(set(broader_group["Resp_ID_DP"])),
    "broader_group Resp_ID_DP unique": broader_group["Resp_ID_DP"].duplicated().sum() == 0,
    "focused_group Resp_ID_DP unique": focused_group["Resp_ID_DP"].duplicated().sum() == 0,
    "every focused-group record satisfies all 5 filters": bool(recheck.all()),
    "occupation classification covers every observed Q14 value": n_unclassified == 0,
    "all 5 product fields fully tokenize (no unresolved fragments)": True,  # asserted in Sec. 3
}
for label, ok in checks.items():
    print(f"  [{'OK' if ok else 'FAIL'}] {label}")
assert all(checks.values())

print(f"\nBroader confirmed salaried Gen Z group (within Mains): n = {len(broader_group):,}")
print(f"Focused group (considers MF, does not currently hold it): n = {len(focused_group):,}")
print("\nSaved:")
print(" -", broader_path)
print(" -", focused_path)

SELECTION FUNNEL
                                                               step  entering  retained  excluded  unknown_or_ambiguous
           1. Completed Mains (MAIN_COMP_STATUS == 'Main Complete')    109430     53357     56073                     0
                                   2. Gen Z (Life_Stage == 'Gen Z')     53357     24576     28781                     0
               3. Salaried, documented (Q14 in SALARIED_DOCUMENTED)     24576      4346     18892                  1338
                 4. Considers mutual funds (Q23A contains MF token)      4346       553      2901                   892
5. Interpretable Q22A_All, no MF token (does not currently hold MF)       553       553         0                     0

OCCUPATION CLASSIFICATION TOTALS (within Mains-complete Gen Z)
Q14
excluded_non_worker             8535
excluded_business               4769
included_salaried_documented    4346
excluded_skilled_worker         2379
excluded_agriculture            1627
ambiguous_

## Summary

Sample validated. Not calculated or done here, on purpose:

- No dashboard charts.
- No barrier ranking or conclusions about which barriers matter most.
- No filtering-out of past MF investors from the focused group (they're flagged, not removed).
- No resolution of the two open items carried into `docs/cohort_definition.md` §6 and `docs/barrier_coverage.md` (the two undocumented "Service (Urban)" occupation categories, and `A15_D15`'s unresolved routing condition).

Next stage (not this one) would use `data/processed/cohort_focused_considered_mf_not_holding.csv` to actually analyze `AA2_DD2`/`AA3_DD3` (and cautiously `AA1_DD1`) barrier content for the 553-respondent focused group.